In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rc('font', size=16)
np.set_printoptions(precision=3)

from week2a_sd import B_Area, s7055_x, s7055_y, Xcp


In [ ]:

stringer_thickness = 0.5e-3
stringer_length = 20e-3     # Total Length of the L shape

Skin_boom_array  = B_Area
Skin_boom_x = s7055_x
Skin_boom_y = s7055_y
num_stringers = 8
num_top_stringers = 4
stringer_indices = np.array([12, 16, 20, 30, 52, 61, 65, 69])
Stringer_boom_array = np.ones(num_stringers)*stringer_thickness*stringer_length
Stringer_boom_x = np.array([s7055_x[i] for i in stringer_indices])
Stringer_boom_y = np.array([s7055_y[i] for i in stringer_indices])
Shear_flow_y = 5 # dummy value for sheet center calculation
# Assume it to be on the shear center

Total_boom_array = Skin_boom_array.copy()

for j, idx in enumerate(stringer_indices):
    Total_boom_array[idx] += Stringer_boom_array[j]



ds_array = np.sqrt((np.diff(Skin_boom_x)**2) + (np.diff(Skin_boom_y)**2))
ds_array = np.append(ds_array, np.sqrt((Skin_boom_x[-1] - Skin_boom_x[0])**2 + (Skin_boom_y[-1] - Skin_boom_y[0])**2))  # Append the last element to make it the same length as Skin_boom_array



# compute the centroid of the cross-section

Centroid_y = (np.sum(Skin_boom_array*Skin_boom_y) + np.sum(Stringer_boom_array*Stringer_boom_y)) / (np.sum(Skin_boom_array) + np.sum(Stringer_boom_array))
Centroid_x = (np.sum(Skin_boom_array*Skin_boom_x) + np.sum(Stringer_boom_array*Stringer_boom_x)) / (np.sum(Skin_boom_array) + np.sum(Stringer_boom_array))

print(f"Centroid of the cross-section: ({Centroid_x:.3f}, {Centroid_y:.3f})")

# Area moment of inertia about the centroidal axis

Skin_boom_y_centroid = Skin_boom_y - Centroid_y
Skin_boom_x_centroid = Skin_boom_x - Centroid_x

Stringer_boom_y_centroid = Stringer_boom_y - Centroid_y
Stringer_boom_x_centroid = Stringer_boom_x - Centroid_x

X_cp_centroid = Xcp - Centroid_x

Centroidal_Ixx = np.sum(Skin_boom_array*Skin_boom_y_centroid**2) + np.sum(Stringer_boom_array*Stringer_boom_y_centroid**2)
Centroidal_Iyy = np.sum(Skin_boom_array*Skin_boom_x_centroid**2) + np.sum(Stringer_boom_array*Stringer_boom_x_centroid**2)
Centroidal_Ixy = np.sum(Skin_boom_array*Skin_boom_x_centroid*Skin_boom_y_centroid) + np.sum(Stringer_boom_array*Stringer_boom_x_centroid*Stringer_boom_y_centroid)

DR = Centroidal_Ixx*Centroidal_Iyy - Centroidal_Ixy**2

print(f"Centroidal Ixx: {Centroidal_Ixx:.3f}, Centroidal Iyy: {Centroidal_Iyy:.3f}, Centroidal Ixy: {Centroidal_Ixy:.3f}, DR: {DR:.3f}")


q_b_array = np.array([0])

print("initialised q_b_array: ", q_b_array)



# not valid in real cases where the wing is twisting, so q_s_0 is not constant along the length of the wing. Therefore, we cannot use the formula for q_s_0 derived from the fact that the integral of q_s over the length of the skin must equal the integral of q_b over the length of the skin. Instead, we need to calculate q_s_0 at each station along the wing, taking into account the twist and other factors that affect the shear flow in the skin.


Centroid of the cross-section: (0.132, 0.006)
Centroidal Ixx: 0.000, Centroidal Iyy: 0.000, Centroidal Ixy: -0.000, DR: 0.000
initialised q_b_array:  [0]


In [10]:
Skin_boom_x_centroid,Skin_boom_y_centroid

(array([ 0.133,  0.132,  0.131,  0.128,  0.125,  0.121,  0.116,  0.11 ,
         0.104,  0.097,  0.09 ,  0.082,  0.073,  0.065,  0.056,  0.046,
         0.037,  0.028,  0.018,  0.008, -0.002, -0.011, -0.021, -0.03 ,
        -0.039, -0.048, -0.057, -0.065, -0.073, -0.081, -0.088, -0.095,
        -0.101, -0.107, -0.112, -0.117, -0.121, -0.124, -0.127, -0.129,
        -0.131, -0.132, -0.132, -0.132, -0.13 , -0.128, -0.125, -0.121,
        -0.117, -0.111, -0.105, -0.098, -0.091, -0.082, -0.074, -0.065,
        -0.055, -0.045, -0.035, -0.024, -0.013, -0.002,  0.008,  0.019,
         0.03 ,  0.04 ,  0.051,  0.06 ,  0.07 ,  0.079,  0.087,  0.095,
         0.102,  0.109,  0.115,  0.12 ,  0.125,  0.128,  0.131,  0.132]),
 array([-6.355e-03, -6.294e-03, -6.079e-03, -5.690e-03, -5.125e-03,
        -4.399e-03, -3.511e-03, -2.459e-03, -1.256e-03,  8.729e-05,
         1.545e-03,  3.092e-03,  4.693e-03,  6.286e-03,  7.833e-03,
         9.333e-03,  1.075e-02,  1.206e-02,  1.325e-02,  1.429e-02,
      

In [14]:
q_b_array = np.zeros(len(Total_boom_array))

q_b_array[0] = Shear_flow_y * Total_boom_array[0] * (Centroidal_Ixy * Skin_boom_x_centroid[0] - Centroidal_Ixx * Skin_boom_y_centroid[0]) / DR

for i in range(1,len(Total_boom_array)):

    q_increment = (
        Shear_flow_y
        * Total_boom_array[i]
        * (
            Centroidal_Ixy * Skin_boom_x_centroid[i]
            - Centroidal_Ixx * Skin_boom_y_centroid[i]
        )
        / DR
    )
 
    q_b_array[i] = q_b_array[i-1] + q_increment

print("Basic shear flow:")
print(q_b_array)
print("len(Total_boom_array): ", len(Total_boom_array))
print("len(q_b_array): ", len(q_b_array))

# Sheer center calculation



Basic shear flow:
[-5.382e-02 -1.586e-01 -3.618e-01 -6.541e-01 -1.026e+00 -1.466e+00
 -1.964e+00 -2.504e+00 -3.065e+00 -3.745e+00 -4.368e+00 -4.980e+00
 -6.941e+00 -7.506e+00 -8.036e+00 -8.522e+00 -9.867e+00 -1.025e+01
 -1.056e+01 -1.081e+01 -1.135e+01 -1.146e+01 -1.150e+01 -1.146e+01
 -1.136e+01 -1.119e+01 -1.096e+01 -1.067e+01 -1.034e+01 -9.961e+00
 -8.398e+00 -7.966e+00 -7.519e+00 -7.067e+00 -6.618e+00 -6.181e+00
 -5.764e+00 -5.373e+00 -4.749e+00 -4.461e+00 -4.219e+00 -4.031e+00
 -3.878e+00 -3.696e+00 -3.444e+00 -3.116e+00 -2.699e+00 -2.198e+00
 -1.622e+00 -9.855e-01 -3.033e-01  4.080e-01  2.975e+00  3.692e+00
  4.385e+00  5.040e+00  5.642e+00  6.175e+00  6.629e+00  6.992e+00
  7.257e+00  7.717e+00  7.775e+00  7.730e+00  7.586e+00  6.890e+00
  6.570e+00  6.176e+00  5.724e+00  4.091e+00  3.569e+00  3.029e+00
  2.486e+00  1.955e+00  1.456e+00  1.006e+00  6.210e-01  3.174e-01
  1.073e-01  1.164e-14]
len(Total_boom_array):  80
len(q_b_array):  80


In [16]:
q_s_0_SC = -np.sum(q_b_array*ds_array)/np.sum(ds_array) # formula for q_s_0 is derived from the fact that the integral of q_s over the length of the skin must equal the integral of q_b over the length of the skin. This is because the total shear flow in the skin must be equal to the total shear flow in the booms, as they are connected and must balance each other out. Therefore, we can calculate q_s_0 by taking the average of q_b over the length of the skin, which is given by the formula above.

q_s_array_SC = q_b_array + q_s_0_SC

print("Shear center shear flow:")
print(q_s_array_SC)

Shear center shear flow:
[ 1.842  1.738  1.534  1.242  0.871  0.43  -0.067 -0.608 -1.169 -1.849
 -2.472 -3.084 -5.045 -5.61  -6.14  -6.626 -7.971 -8.349 -8.664 -8.912
 -9.458 -9.565 -9.6   -9.564 -9.459 -9.289 -9.059 -8.774 -8.44  -8.065
 -6.501 -6.07  -5.623 -5.171 -4.722 -4.285 -3.868 -3.477 -2.853 -2.565
 -2.322 -2.135 -1.982 -1.8   -1.548 -1.219 -0.803 -0.302  0.274  0.911
  1.593  2.304  4.871  5.588  6.281  6.936  7.538  8.072  8.525  8.888
  9.153  9.614  9.671  9.626  9.482  8.787  8.466  8.072  7.62   5.987
  5.465  4.926  4.382  3.852  3.352  2.902  2.517  2.214  2.003  1.896]


In [17]:
# ---------------------------------------------------------
# Moment of shear flow about reference point O
# ---------------------------------------------------------

x0 = 0.0
y0 = 0.0

x = Skin_boom_x - x0
y = Skin_boom_y - y0

x_next = np.roll(x, -1)
y_next = np.roll(y, -1)

dx = x_next - x
dy = y_next - y

M_q = np.sum(
    q_s_array_SC * (
        x * dy - y * dx
    )
)

print(f"Moment of shear flow about O = {M_q:.6e}")

Moment of shear flow about O = -3.272729e-02


In [19]:
# ---------------------------------------------------------
# Shear center
# ---------------------------------------------------------

xi_S = M_q / Shear_flow_y

print(f"Shear center location from O = {xi_S:.6e} m")

Shear center location from O = -6.545458e-03 m
